In [1]:
import os
import pandas as pd
import requests
import json
from time import sleep
from bs4 import BeautifulSoup

from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Disable SSL warnings
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

from SentimentAnalysis import sentiment_from_api, individual_sentiment_from_api
from ProminenceAnalysis import check_name_appearance
from Keyword import keyword_occurrences
from Tier import tier_online


In [2]:
# The function to log into a specified domain
def log_in_xxx_domain(driver, url): 
    """
    Automates the login process for specific Australian news websites.

    Parameters:
    driver: Selenium WebDriver instance used to automate browser interactions.
    url (str): The URL of the news website to log into.

    The function performs the following steps:
    1. Navigates to the given URL.
    2. Waits for 10 seconds to ensure the page is fully loaded.
    3. Based on the URL, executes a specific login procedure for the respective news website.
       - For 'theaustralian.com.au', 'heraldsun.com.au', 'dailytelegraph':
         - Finds and clicks the login button.
         - Waits for and fills in the email and password fields.
         - Clicks the submit button to complete the login.
       - For 'smh.com.au':
         - Similar process with different element selectors.
       - For 'afr.com':
         - Again, a similar process with different element selectors.
    4. After attempting to log in, waits for 5 seconds and reloads the page.
    5. If a TimeoutException or NoSuchElementException occurs, prints an error message.

    The function returns the driver instance, which can be used for further automated tasks 
    on the website after login.

    Note:
    - The function uses hardcoded credentials for login, which is not a recommended practice 
      for production code due to security concerns.
    - The function assumes that the driver is correctly configured and that the elements 
      necessary for login are present and correctly identified in the page's DOM.
    """
    
    # Navigate the driver to the specified URL.
    page = driver.get(url) 
    # Wait for 10 seconds for the page to load.
    sleep(10) 

    try:
        # Check if the URL belongs to a specific set of Australian news websites.
        if "www.theaustralian.com.au" in url or "www.heraldsun.com.au" in url or "www.dailytelegraph" in url:
            # Find the Log in button and click it
            buttons = driver.find_elements(By.CSS_SELECTOR, "button.log-in.dsf-user-login")
            if not buttons:
                return driver
            buttons[0].click()
        
            # Find and fill in the email input box.
            email_box = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, "//input[@type='email'][@id='1-email'][@name='email']"))
            )
            email_box.send_keys("finance@chepnetwork.com.au")
        
            # Find and fill in the password input box.
            password_box = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, "//input[@type='password'][@name='password'][@class='auth0-lock-input']"))
            )
            password_box.send_keys("predge")
            
            sleep(5) # Wait for 5 seconds
        
            # Click the login button
            login_button = driver.find_element(By.XPATH, "//button[@class='auth0-lock-submit'][@name='submit'][@type='submit']")
            login_button.click()
            
        # Repeat similar process
        elif "www.smh.com.au" in url:
            # Find the Log in link and click it
            login_links = driver.find_elements(By.CSS_SELECTOR, "a[data-testid='login-button-myaccount']")
            if not login_links:
                return driver
            login_links[0].click()
        
            # Enter email
            email_box = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, "//input[@type='email'][@id='login-email-address'][@name='emailAddress']"))
            )
            email_box.send_keys("finance@cheproximity.com.au")
        
            # Enter password
            password_box = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.ID, "login-password"))
            )
            password_box.send_keys("predge")
            
            sleep(5) 
        
            # Click the login button
            login_button = driver.find_element(By.XPATH, "//button[@data-testid='login-button']")
            login_button.click()
    
        elif "www.afr.com" in url:
            # Find the Log in span and click it
            login_links = driver.find_elements(By.XPATH, "//span[text()='Log in']")
            if not login_links:
                return driver
            login_links[0].click()
        
            # Enter email
            email_box = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, "//input[@type='email'][@id='loginEmail'][@data-testid='LoginEmailAddress-input']"))
            )
            email_box.send_keys("finance@cheproximity.com.au")
        
            # Enter password
            password_box = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, "//input[@type='password'][@id='loginPassword'][@data-testid='LoginPassword-input']"))
            )
            password_box.send_keys("predge")
            
            sleep(5) 
        
            # Click the login button
            login_button = driver.find_element(By.XPATH, "//button[@class='_2W8wR'][@data-testid='LoginPassword-submit'][@type='submit']")
            login_button.click()

        sleep(5) # Wait for 5 seconds after login attempts
        driver.get(url) # Reload the page

    except (TimeoutException, NoSuchElementException):
        # Print an error message if the login process fails due to 
        # a timeout or missing element.
        print(f"Failed to log in to {url}")
    
    return driver
        

In [3]:
#driver = webdriver.Chrome()
#url = 'https://www.dailytelegraph.com.au/subscribe/news/1/?mode=aod&offerset=dt_default&utm_medium=SEM&int_source=SEM&int_campaign=2024_Metros_ExternalDigital&int_content=SEM&sourceCode=DTWEB_SEM20240703&gclid=CjwKCAjw-eKpBhAbEiwAqFL0mv9MlTCHFUzc8e0OcJZVo9_1pejdFzWRaIn6ETXgT24WaPW-JUepHhoCgsQQAvD_BwE&gclsrc=aw.ds' 
#log_in_xxx_domain(driver, url)

In [7]:
# The function to extract text from a given URL and save it in a specified folder.
def extract_text_from_url(url, folder_name):
    """
    Retrieves text from a webpage and saves it to a specified folder.

    Parameters:
    url (str): The URL of the webpage from which text is to be extracted.
    folder_name (str): The name of the folder where the extracted text will be saved.

    The function performs the following steps:
    1. Sends a GET request to the URL.
    2. Checks the response status. If it's not 200 (OK), prints an error message and returns None.
    3. Parses the webpage content using BeautifulSoup.
    4. Removes any script and style elements from the parsed content.
    5. Extracts the text, removes leading/trailing whitespace, and drops blank lines.
    6. Converts the cleaned text to lowercase.
    7. Ensures the specified folder exists, or creates it if it does not.
    8. Saves the JSON response from the request to 'response.json' in the specified folder.
    9. Saves the cleaned text to a file in the specified folder.
    10. Returns the cleaned text.

    Exception Handling:
    - Catches and prints a message if a timeout occurs during the GET request.

    Note:
    - The function uses two separate requests to the same URL, which is redundant and could be optimized.
    - There's a potential security risk in setting 'verify = False' in the GET request, as it 
      bypasses SSL certificate verification.
    - The function assumes the URL returns a JSON response, which may not always be the case.
    - The folder name for saving the cleaned text is used as the filename, which could cause 
      issues. The filename should be explicitly specified.
    """
    
    try:
        # Send a GET request to the webpage with a timeout of 10 seconds.
        response = requests.get(url, timeout = 10.0)
        # Additional error handling for other issues can be added here.

        # Check if the request was successful (status code 200).
        if response.status_code != 200:
            print(f"Failed to retrieve the webpage. Status code: {response.status_code}")
            print(url)
            return None
        
        # Parse the HTML content of the page using BeautifulSoup.
        soup = BeautifulSoup(response.content, 'html.parser')
    
        # Remove any script and style elements from the HTML.
        for script in soup(["script", "style"]):
            script.decompose()
    
        # Extract the text content from the HTML.
        text = soup.get_text()
    
        # Break the text into lines and remove any leading/trailing whitespace
        lines = (line.strip() for line in text.splitlines())
        
        # Drop blank lines and join lines to form the final text
        cleaned_text = "\n".join(line for line in lines if line)

        # Convert the cleaned_text to lowercase
        cleaned_text = cleaned_text.lower()
            
        # Ensure the directory exists before saving the file
        if not os.path.exists(folder_name):
            os.makedirs(folder_name)
        
        # Save the cleaned_text to the specified file
        with open(folder_name, 'w', encoding = 'utf-8') as file:
            file.write(cleaned_text)
    
    except requests.Timeout as err:
        # Print the error if a timeout occurs during the request.
        print(err)
    
    return cleaned_text
    

In [9]:
if __name__ == '__main__':
    
    table = pd.read_excel('Tracker2023.xlsx', skiprows = 1)
    # Initialize an empty dictionary
    record = {}

    urls = table.loc[table['MEDIUM'].str.lower() == 'online']['URL'].to_list()

    # Ensure 'downloads_online' directory exists or create it
    if not os.path.exists('downloads_online'):
        os.makedirs('downloads_online')
    
    for each_index, each_url in enumerate(tqdm(urls)):
        # Process each URL and store results
        filename = f'{each_index}'
        folder_name = f'downloads_online'
        file_path = os.path.join(folder_name, filename)
        content = extract_text_from_url(each_url, file_path)
        if content:
            
            #sentiment = sentiment_from_api(content, folder_name) if content else None
    
            company_name_variants = ['flybuys', 'onepass']
            appearance_data = check_name_appearance(content, company_name_variants)
         
            record[filename] = {'content': content, **appearance_data}
            
            with open(f'{filename}_result.json', 'w', encoding = 'utf-8') as fileout:
                json.dump(record, fileout, indent = 4, ensure_ascii = False)

 36%|███████████████▎                           | 10/28 [00:09<00:18,  1.05s/it]

Failed to retrieve the webpage. Status code: 404
https://au.finance.yahoo.com/news/flybuys-major-change-for-kmart-target-bunnings-and-officeworks-shoppers-050200349.html


100%|███████████████████████████████████████████| 28/28 [00:47<00:00,  1.71s/it]
